# Probability & Statistics: Full Workflow Demo

This notebook walks through a complete statistical analysis workflow:
1. Descriptive Statistics — understand your data's shape and spread
2. Distribution Fitting — find the best parametric model
3. Hypothesis Testing — make principled inferences
4. Power Analysis — plan experiments correctly
5. Bayesian A/B Testing — probabilistic decision-making
6. SQL Analytics (DuckDB) — scalable statistical queries

All configuration comes from `config.yaml`.

In [ ]:
import sys
import pathlib

# Add project root to path
project_root = pathlib.Path('..').resolve().parent
sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('inline')
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import yaml
import warnings
warnings.filterwarnings('ignore')

# Load config
config_path = project_root / 'config.yaml'
with open(config_path) as f:
    cfg = yaml.safe_load(f)

# Ensure output directories exist
pathlib.Path(cfg['data']['output_dir']).mkdir(exist_ok=True)
pathlib.Path('logs').mkdir(exist_ok=True)

print('Config loaded:', list(cfg.keys()))
rng = np.random.default_rng(cfg['data']['random_seed'])
N = cfg['data']['sample_size']
print(f'Sample size: {N}, Random seed: {cfg["data"]["random_seed"]}')

## 1. Descriptive Statistics

### The Central Limit Theorem (CLT)
Regardless of the underlying distribution, the sampling distribution of the **mean** approaches normality as n → ∞.

We demonstrate this by sampling from a skewed chi-squared distribution and observing that sample means are normally distributed.

In [ ]:
# Generate three datasets
normal_data   = rng.normal(loc=50, scale=10, size=N)
skewed_data   = rng.chisquare(df=3, size=N)  # positively skewed
bimodal_data  = np.concatenate([
    rng.normal(30, 5, N//2),
    rng.normal(70, 5, N//2)
])

# Comprehensive descriptive statistics
def compute_stats(data, label):
    trimmed = stats.trim_mean(data, cfg['descriptive_stats']['trimmed_mean_pct'])
    percentiles = np.quantile(data, cfg['descriptive_stats']['percentiles'])
    return {
        'label': label, 'n': len(data),
        'mean': np.mean(data), 'median': np.median(data),
        'trimmed_mean': trimmed, 'std': np.std(data, ddof=1),
        'iqr': stats.iqr(data),
        'mad': np.median(np.abs(data - np.median(data))),
        'cv': np.std(data, ddof=1) / np.mean(data),
        'skewness': stats.skew(data), 'kurtosis': stats.kurtosis(data),
        'P25': percentiles[1], 'P50': percentiles[2],
        'P75': percentiles[3], 'P99': percentiles[-1]
    }

summary_df = pd.DataFrame([
    compute_stats(normal_data,  'Normal(50,10)'),
    compute_stats(skewed_data,  'Chi2(df=3)'),
    compute_stats(bimodal_data, 'Bimodal'),
])
display(summary_df[['label','mean','median','trimmed_mean','std','iqr','skewness','kurtosis']].round(3))

In [ ]:
# Visualise distributions
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

datasets = [
    (normal_data,  'Normal(50,10)',  'steelblue'),
    (skewed_data,  'Chi2(df=3)',     'firebrick'),
    (bimodal_data, 'Bimodal',        'seagreen'),
]

for col, (data, label, color) in enumerate(datasets):
    # Histogram
    axes[0, col].hist(data, bins=cfg['descriptive_stats']['distribution_bins'],
                      color=color, alpha=0.7, edgecolor='white', density=True)
    axes[0, col].axvline(np.mean(data), color='black', lw=2, linestyle='-', label='Mean')
    axes[0, col].axvline(np.median(data), color='orange', lw=2, linestyle='--', label='Median')
    axes[0, col].set_title(f'{label}\nskew={stats.skew(data):.2f}, kurt={stats.kurtosis(data):.2f}')
    axes[0, col].legend(fontsize=8)
    axes[0, col].set_xlabel('Value'); axes[0, col].set_ylabel('Density')

    # Box plot with outlier marking
    axes[1, col].boxplot(data, vert=True, patch_artist=True,
                         boxprops=dict(facecolor=color, alpha=0.5),
                         medianprops=dict(color='black', linewidth=2))
    axes[1, col].set_title(f'{label} — Box Plot')
    axes[1, col].set_ylabel('Value')

plt.suptitle('Distribution Comparison: Histogram & Box Plot', fontsize=13)
plt.tight_layout()
plt.savefig(cfg['data']['output_dir'] + 'notebook_distributions.png', dpi=150)
plt.show()

In [ ]:
# Central Limit Theorem demonstration
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
sample_sizes = [5, 10, 30, 100]
n_sim = 5000

for ax, n in zip(axes, sample_sizes):
    sample_means = [np.mean(rng.chisquare(df=3, size=n)) for _ in range(n_sim)]
    ax.hist(sample_means, bins=50, density=True, color='steelblue',
            alpha=0.7, edgecolor='white')
    # Overlay normal curve
    mu_hat, std_hat = np.mean(sample_means), np.std(sample_means)
    x = np.linspace(mu_hat - 4*std_hat, mu_hat + 4*std_hat, 300)
    ax.plot(x, stats.norm.pdf(x, mu_hat, std_hat), 'firebrick', lw=2)
    sw_stat, sw_p = stats.shapiro(sample_means[:200])  # Shapiro on subset
    ax.set_title(f'n={n}\nShapiro p={sw_p:.3f}')
    ax.set_xlabel('Sample Mean')

plt.suptitle('Central Limit Theorem: Chi²(df=3) Sample Means', fontsize=12)
plt.tight_layout()
plt.savefig(cfg['data']['output_dir'] + 'notebook_clt.png', dpi=150)
plt.show()
print('As n increases, sample means converge to Normal regardless of the original distribution.')

## 2. Distribution Fitting

We fit multiple parametric distributions and select the best using **AIC** (Akaike Information Criterion).

**AIC = 2k - 2·log(L)** — penalises model complexity (k parameters) relative to fit (log-likelihood L). Lower AIC is better.

In [ ]:
# Fit distributions to log-normal data
lognormal_data = rng.lognormal(mean=0, sigma=0.8, size=N)

candidate_dists = {
    'normal':      stats.norm,
    'lognormal':   stats.lognorm,
    'exponential': stats.expon,
    'gamma':       stats.gamma,
}

fit_results = []
for name, dist in candidate_dists.items():
    params = dist.fit(lognormal_data)
    k = len(params)
    log_like = np.sum(dist.logpdf(lognormal_data, *params))
    aic = 2*k - 2*log_like
    ks_stat, ks_p = stats.kstest(lognormal_data, name, args=params)
    fit_results.append({'Distribution': name, 'Params': params,
                        'AIC': round(aic,2), 'KS stat': round(ks_stat,4),
                        'KS p-value': round(ks_p,4)})

fit_df = pd.DataFrame(fit_results).sort_values('AIC')
display(fit_df[['Distribution','AIC','KS stat','KS p-value']])

In [ ]:
# Plot best fit and QQ plot
best = fit_df.iloc[0]
best_name = best['Distribution']
best_dist = candidate_dists[best_name]
best_params = best['Params']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Histogram + PDF
x = np.linspace(lognormal_data.min(), lognormal_data.max(), 500)
axes[0].hist(lognormal_data, bins=50, density=True, color='steelblue', alpha=0.6, edgecolor='white')
axes[0].plot(x, best_dist.pdf(x, *best_params), 'firebrick', lw=2,
             label=f'{best_name} (AIC={best["AIC"]:.1f})')
axes[0].set_title('Best Fit: Histogram + PDF')
axes[0].legend(); axes[0].set_xlabel('Value')

# ECDF vs Theoretical CDF
sorted_d = np.sort(lognormal_data)
ecdf = np.arange(1, N+1) / N
tcdf = best_dist.cdf(sorted_d, *best_params)
axes[1].step(sorted_d, ecdf, where='post', color='steelblue', lw=2, label='ECDF')
axes[1].plot(sorted_d, tcdf, 'firebrick', lw=2, ls='--', label=f'Fitted CDF')
axes[1].set_title('ECDF vs Theoretical CDF'); axes[1].legend()

# QQ plot (normal)
(osm, osr), (slope, intercept, r) = stats.probplot(np.log(lognormal_data), dist='norm')
axes[2].plot(osm, osr, 'o', ms=3, color='steelblue', alpha=0.5)
axes[2].plot([osm[0], osm[-1]], [slope*osm[0]+intercept, slope*osm[-1]+intercept],
             'firebrick', lw=2)
axes[2].set_title(f'Normal QQ Plot of log(data)\nR²={r**2:.4f}')
axes[2].set_xlabel('Theoretical Quantiles'); axes[2].set_ylabel('Sample Quantiles')

plt.suptitle(f'Distribution Fitting: Log-Normal Data — Best Fit: {best_name}', fontsize=12)
plt.tight_layout()
plt.savefig(cfg['data']['output_dir'] + 'notebook_dist_fitting.png', dpi=150)
plt.show()

## 3. Hypothesis Testing

### Understanding p-values
A **p-value** is the probability of observing data at least as extreme as what was observed, *assuming H₀ is true*.

- p < α → reject H₀ (statistically significant)
- p ≥ α → fail to reject H₀ (insufficient evidence)

**It does NOT mean the probability that H₀ is true!**

In [ ]:
alpha = cfg['hypothesis_testing']['alpha']

# Simulate control and treatment groups
control   = rng.normal(50, 10, 500)
treatment = rng.normal(52.5, 10, 500)  # true effect = 2.5

# 1. Welch's two-sample t-test
t_stat, p_val = stats.ttest_ind(control, treatment, equal_var=False)
cohen_d = (np.mean(treatment) - np.mean(control)) / np.sqrt(
    (np.var(control, ddof=1) + np.var(treatment, ddof=1)) / 2
)

print('=== Welch Two-sample t-test ===')
print(f'  Control:   mean={np.mean(control):.2f}, std={np.std(control,ddof=1):.2f}, n={len(control)}')
print(f'  Treatment: mean={np.mean(treatment):.2f}, std={np.std(treatment,ddof=1):.2f}, n={len(treatment)}')
print(f'  t={t_stat:.4f}, p={p_val:.6f}')
print(f'  Cohen\'s d={cohen_d:.4f} (small=0.2, medium=0.5, large=0.8)')
print(f'  Decision: {"REJECT H0" if p_val < alpha else "FAIL TO REJECT H0"} (alpha={alpha})')

# 2. Mann-Whitney U (non-parametric)
u_stat, u_p = stats.mannwhitneyu(control, treatment, alternative='two-sided')
r_biserial = 1 - 2 * u_stat / (len(control) * len(treatment))
print(f'\n=== Mann-Whitney U (non-parametric) ===')
print(f'  U={u_stat:.1f}, p={u_p:.6f}, rank-biserial r={r_biserial:.4f}')

# 3. Chi-squared test
contingency = np.array([[200, 150], [80, 220]])
chi2, chi2_p, dof, expected = stats.chi2_contingency(contingency)
n_ct = contingency.sum()
cramers_v = np.sqrt(chi2 / (n_ct * (min(contingency.shape)-1)))
print(f'\n=== Chi-squared Independence Test ===')
print(f'  Observed table: {contingency.tolist()}')
print(f'  chi2={chi2:.4f}, p={chi2_p:.6f}, dof={dof}, Cramér\'s V={cramers_v:.4f}')

In [ ]:
# Visualise: distribution of t-statistics under H0 vs observed
df_val = len(control) + len(treatment) - 2
t_range = np.linspace(-5, 5, 1000)
t_pdf = stats.t.pdf(t_range, df=df_val)

critical = stats.t.ppf(1 - alpha/2, df=df_val)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# t-distribution
axes[0].plot(t_range, t_pdf, 'steelblue', lw=2, label='t-distribution under H₀')
axes[0].fill_between(t_range, t_pdf, where=t_range > critical, color='firebrick',
                     alpha=0.4, label=f'Rejection region (alpha/2={alpha/2})')
axes[0].fill_between(t_range, t_pdf, where=t_range < -critical, color='firebrick', alpha=0.4)
axes[0].axvline(t_stat, color='black', lw=2, linestyle='--',
                label=f'Observed t={t_stat:.3f}')
axes[0].set_title(f'Two-sample t-test\np={p_val:.5f}, d={cohen_d:.3f}')
axes[0].legend(fontsize=8); axes[0].set_xlabel('t-statistic')

# Bootstrap CI for the mean difference
n_boot = cfg['hypothesis_testing']['bootstrap_iterations']
boot_diffs = np.array([
    np.mean(rng.choice(treatment, len(treatment), replace=True)) -
    np.mean(rng.choice(control, len(control), replace=True))
    for _ in range(n_boot)
])
ci_lo, ci_hi = np.percentile(boot_diffs, [2.5, 97.5])
obs_diff = np.mean(treatment) - np.mean(control)

axes[1].hist(boot_diffs, bins=50, density=True, color='steelblue', alpha=0.7, edgecolor='white')
axes[1].axvline(obs_diff, color='black', lw=2, label=f'Observed diff={obs_diff:.3f}')
axes[1].axvline(ci_lo, color='firebrick', lw=2, linestyle='--', label=f'95% CI: ({ci_lo:.3f}, {ci_hi:.3f})')
axes[1].axvline(ci_hi, color='firebrick', lw=2, linestyle='--')
axes[1].axvline(0, color='grey', lw=1.5, linestyle=':')
axes[1].set_title(f'Bootstrap CI (B={n_boot:,}): Mean Difference')
axes[1].legend(fontsize=8); axes[1].set_xlabel('Mean(Treatment) - Mean(Control)')

plt.suptitle('Hypothesis Testing Visualisations', fontsize=12)
plt.tight_layout()
plt.savefig(cfg['data']['output_dir'] + 'notebook_hypothesis_tests.png', dpi=150)
plt.show()

## 4. Power Analysis

**Statistical power** = P(reject H₀ | H₁ is true) = 1 − β

Power depends on:
- **Effect size** (d): larger effects are easier to detect
- **Sample size** (n): more data → higher power
- **Significance level** (α): lower α → lower power

The classic target is **80% power** at **α = 0.05**.

In [ ]:
def achieved_power(n, d, alpha=0.05):
    """Power of a two-sample t-test given n per group and Cohen's d."""
    z_a = stats.norm.ppf(1 - alpha/2)
    ncp = abs(d) * np.sqrt(n / 2)
    return stats.norm.cdf(ncp - z_a)

def sample_size_ttest(d, alpha=0.05, power=0.80):
    z_a = stats.norm.ppf(1 - alpha/2)
    z_b = stats.norm.ppf(power)
    return int(np.ceil(2 * (z_a + z_b)**2 / d**2))

# Sample size table
print('=== Sample Size Requirements (80% power, alpha=0.05) ===')
print(f'{"Cohen\'s d":<12} {"n per group":<15} {"Total N"}')
for d in [0.10, 0.20, 0.30, 0.50, 0.80, 1.00]:
    n = sample_size_ttest(d)
    effect_label = 'tiny' if d < 0.2 else ('small' if d < 0.5 else ('medium' if d < 0.8 else 'large'))
    print(f'd={d:.2f} ({effect_label:<7}) {n:<15} {2*n}')

In [ ]:
# Power curve visualisation
n_vals = np.arange(10, 501, 5)
effect_sizes = [0.2, 0.3, 0.5, 0.8]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(effect_sizes)))
for d, color in zip(effect_sizes, colors):
    powers = [achieved_power(n, d) for n in n_vals]
    axes[0].plot(n_vals, powers, color=color, lw=2, label=f'd={d:.1f}')

axes[0].axhline(cfg['hypothesis_testing']['power'], color='firebrick',
                ls='--', lw=1.5, label=f'Target power={cfg["hypothesis_testing"]["power"]}')
axes[0].set_xlabel('n per group'); axes[0].set_ylabel('Power (1 - β)')
axes[0].set_title('Power Curves: Two-sample t-test')
axes[0].legend(); axes[0].set_ylim(0, 1)
axes[0].grid(alpha=0.3)

# Alpha-power tradeoff heatmap
alpha_vals = [0.001, 0.005, 0.01, 0.05, 0.10]
power_vals = [0.70, 0.80, 0.90, 0.95]
n_matrix = np.array([
    [sample_size_ttest(0.5, a, p) for a in alpha_vals]
    for p in power_vals
])
im = axes[1].imshow(n_matrix, cmap='YlOrRd', aspect='auto')
axes[1].set_xticks(range(len(alpha_vals)))
axes[1].set_xticklabels([f'{a}' for a in alpha_vals])
axes[1].set_yticks(range(len(power_vals)))
axes[1].set_yticklabels([f'{p:.2f}' for p in power_vals])
axes[1].set_xlabel('Alpha'); axes[1].set_ylabel('Power')
axes[1].set_title('Required n per group (Cohen\'s d=0.5)')
for i in range(len(power_vals)):
    for j in range(len(alpha_vals)):
        axes[1].text(j, i, str(int(n_matrix[i, j])), ha='center', va='center', fontsize=9)
plt.colorbar(im, ax=axes[1])

plt.tight_layout()
plt.savefig(cfg['data']['output_dir'] + 'notebook_power_analysis.png', dpi=150)
plt.show()

## 5. Bayesian A/B Testing

### Beta-Binomial Conjugate Model

**Prior**: p ~ Beta(α₀, β₀)

**Likelihood**: X ~ Binomial(n, p)

**Posterior**: p | X ~ Beta(α₀ + successes, β₀ + failures)

The posterior mean is a weighted combination of the prior mean and observed rate:
- With more data, the posterior converges to the observed rate
- With less data, the prior belief dominates

In [ ]:
# Bayesian A/B test
bay_cfg = cfg['bayesian_ab']
prior_a, prior_b = bay_cfg['prior_alpha'], bay_cfg['prior_beta']
mc_samples = bay_cfg['monte_carlo_samples']

# Observed data
n_a, conv_a = 1000, 100
n_b, conv_b = 1000, 130

# Posterior parameters
al_a = prior_a + conv_a
be_a = prior_b + (n_a - conv_a)
al_b = prior_a + conv_b
be_b = prior_b + (n_b - conv_b)

# Monte Carlo: P(B > A)
mc_rng = np.random.default_rng(cfg['data']['random_seed'])
samples_a = mc_rng.beta(al_a, be_a, mc_samples)
samples_b = mc_rng.beta(al_b, be_b, mc_samples)
prob_b_gt_a = np.mean(samples_b > samples_a)

# Expected loss
loss_a = np.mean(np.maximum(0, samples_b - samples_a))  # loss from choosing A
loss_b = np.mean(np.maximum(0, samples_a - samples_b))  # loss from choosing B

# Credible intervals
ci_level = bay_cfg['credible_interval']
lo = (1 - ci_level) / 2
ci_a = (stats.beta.ppf(lo, al_a, be_a), stats.beta.ppf(1-lo, al_a, be_a))
ci_b = (stats.beta.ppf(lo, al_b, be_b), stats.beta.ppf(1-lo, al_b, be_b))

# Frequentist comparison
p1, p2 = conv_a/n_a, conv_b/n_b
p_pool = (conv_a + conv_b) / (n_a + n_b)
se_freq = np.sqrt(p_pool*(1-p_pool)*(1/n_a + 1/n_b))
z_freq = (p2 - p1) / se_freq
p_freq = 2*(1 - stats.norm.cdf(abs(z_freq)))

print('=== Bayesian A/B Test ===')
print(f'Group A: {conv_a}/{n_a} = {p1:.4f}')
print(f'Group B: {conv_b}/{n_b} = {p2:.4f}')
print(f'Posterior mean A: {al_a/(al_a+be_a):.4f}, {int(ci_level*100)}% CI: ({ci_a[0]:.4f}, {ci_a[1]:.4f})')
print(f'Posterior mean B: {al_b/(al_b+be_b):.4f}, {int(ci_level*100)}% CI: ({ci_b[0]:.4f}, {ci_b[1]:.4f})')
print(f'P(B > A) = {prob_b_gt_a:.4f} ({mc_samples:,} MC samples)')
print(f'Expected loss(A) = {loss_a:.6f}')
print(f'Expected loss(B) = {loss_b:.6f}')
print(f'Frequentist: z={z_freq:.4f}, p={p_freq:.6f}')

# Decision
if prob_b_gt_a > 0.95:
    print('\nDecision: SHIP B (Bayesian probability > 95%)')
elif prob_b_gt_a < 0.05:
    print('\nDecision: SHIP A (Bayesian probability > 95%)')
else:
    print('\nDecision: INCONCLUSIVE — collect more data')

In [ ]:
# Bayesian visualisations
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Posterior distributions
x = np.linspace(0.05, 0.25, 1000)
pdf_a = stats.beta.pdf(x, al_a, be_a)
pdf_b = stats.beta.pdf(x, al_b, be_b)
axes[0].plot(x, pdf_a, 'steelblue', lw=2, label='Posterior A')
axes[0].fill_between(x, pdf_a, alpha=0.2, color='steelblue')
axes[0].plot(x, pdf_b, 'firebrick', lw=2, label='Posterior B')
axes[0].fill_between(x, pdf_b, alpha=0.2, color='firebrick')
axes[0].axvline(p1, color='steelblue', ls=':', alpha=0.8)
axes[0].axvline(p2, color='firebrick', ls=':', alpha=0.8)
axes[0].set_title(f'Posterior Distributions\nP(B>A) = {prob_b_gt_a:.4f}')
axes[0].set_xlabel('Conversion Rate'); axes[0].legend()

# 2. Lift distribution (Monte Carlo)
lift = samples_b - samples_a
axes[1].hist(lift, bins=80, density=True, color='steelblue', alpha=0.7, edgecolor='white')
axes[1].axvline(0, color='black', lw=1.5, ls='--', label='No effect')
axes[1].axvline(np.mean(lift), color='firebrick', lw=2,
                label=f'Mean lift={np.mean(lift):.4f}')
lo_lift, hi_lift = np.percentile(lift, [2.5, 97.5])
axes[1].axvline(lo_lift, color='darkorange', ls=':', label=f'95% CI: ({lo_lift:.3f}, {hi_lift:.3f})')
axes[1].axvline(hi_lift, color='darkorange', ls=':')
axes[1].set_title('Lift Distribution (rate_B - rate_A)')
axes[1].set_xlabel('Lift'); axes[1].legend(fontsize=8)

# 3. Sequential Bayesian updating (P(B>A) over time)
seq_rng = np.random.default_rng(42)
true_a, true_b = 0.10, 0.13
probs_over_time = []
ns = []
ca, na_seq = 0, 0
cb, nb_seq = 0, 0
for i in range(1, 1001):
    ca += seq_rng.binomial(1, true_a)
    cb += seq_rng.binomial(1, true_b)
    na_seq += 1; nb_seq += 1
    if i % 10 == 0:
        s_a = seq_rng.beta(prior_a + ca, prior_b + (na_seq - ca), 10000)
        s_b = seq_rng.beta(prior_a + cb, prior_b + (nb_seq - cb), 10000)
        probs_over_time.append(np.mean(s_b > s_a))
        ns.append(i)

axes[2].plot(ns, probs_over_time, 'steelblue', lw=2)
axes[2].axhline(0.95, color='firebrick', ls='--', lw=1.5, label='Stop at 0.95')
axes[2].axhline(0.5, color='grey', ls=':', lw=1)
axes[2].set_title(f'Sequential: P(B>A) — true rates {true_a} vs {true_b}')
axes[2].set_xlabel('N per group'); axes[2].set_ylabel('P(B > A)')
axes[2].set_ylim(0, 1); axes[2].legend()

plt.suptitle('Bayesian A/B Testing', fontsize=12)
plt.tight_layout()
plt.savefig(cfg['data']['output_dir'] + 'notebook_bayesian_ab.png', dpi=150)
plt.show()

In [ ]:
# Prior sensitivity: how informative priors shift the posterior
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
x = np.linspace(0.001, 0.5, 1000)
n_obs, s_obs = 30, 5  # 5 successes in 30 trials (16.7%)

prior_specs = [
    (1, 1,   'Uniform (vague)'),
    (2, 18,  'Weakly informative (10% prior belief)'),
    (10, 90, 'Strongly informative (10% prior, high certainty)'),
]

for ax, (pa, pb, label) in zip(axes, prior_specs):
    prior_pdf = stats.beta.pdf(x, pa, pb)
    post_pdf  = stats.beta.pdf(x, pa + s_obs, pb + (n_obs - s_obs))

    ax.plot(x, prior_pdf, 'grey', lw=2, ls='--', label='Prior')
    ax.plot(x, post_pdf, 'steelblue', lw=2, label='Posterior')
    ax.axvline(s_obs/n_obs, color='firebrick', ls=':', lw=2, label=f'MLE={s_obs/n_obs:.3f}')
    ax.set_title(label); ax.set_xlabel('p'); ax.legend(fontsize=8)

plt.suptitle(f'Prior Sensitivity: Beta-Binomial | {s_obs}/{n_obs} observed', fontsize=12)
plt.tight_layout()
plt.savefig(cfg['data']['output_dir'] + 'notebook_prior_sensitivity.png', dpi=150)
plt.show()

## 6. SQL Analytics with DuckDB

DuckDB enables SQL-based statistical analysis directly on DataFrames — no database server required.

SQL statistical functions demonstrated:
- `PERCENTILE_CONT` — exact percentile calculation
- `STDDEV_SAMP`, `VAR_SAMP` — sample statistics
- `PERCENT_RANK`, `CUME_DIST`, `NTILE` — ranking window functions
- `WIDTH_BUCKET` — histogram binning
- `REGR_SLOPE`, `REGR_R2` — SQL linear regression

In [ ]:
import duckdb

# Generate e-commerce data
rng2 = np.random.default_rng(cfg['data']['random_seed'])
n_txn = cfg['data']['sample_size']

revenues = np.abs(rng2.lognormal(3.5, 1.2, n_txn))
categories = rng2.choice(['Electronics','Clothing','Books','Home','Sports'],
                         n_txn, p=[0.25,0.30,0.15,0.20,0.10])

txn_df = pd.DataFrame({'revenue': revenues, 'category': categories})
con = duckdb.connect()
con.register('transactions', txn_df)

# SQL descriptive statistics
desc_sql = """
SELECT
    COUNT(revenue)                                         AS n,
    ROUND(AVG(revenue), 2)                                 AS mean,
    ROUND(STDDEV_SAMP(revenue), 2)                         AS std_dev,
    ROUND(PERCENTILE_CONT(0.25) WITHIN GROUP
        (ORDER BY revenue), 2)                             AS q1,
    ROUND(PERCENTILE_CONT(0.50) WITHIN GROUP
        (ORDER BY revenue), 2)                             AS median,
    ROUND(PERCENTILE_CONT(0.75) WITHIN GROUP
        (ORDER BY revenue), 2)                             AS q3,
    ROUND(PERCENTILE_CONT(0.99) WITHIN GROUP
        (ORDER BY revenue), 2)                             AS p99,
    ROUND(SKEWNESS(revenue), 3)                            AS skewness,
    ROUND(KURTOSIS(revenue), 3)                            AS kurtosis
FROM transactions
"""
print('=== SQL Descriptive Statistics ===')
display(con.execute(desc_sql).df())

In [ ]:
# SQL: IQR outlier detection
outlier_sql = f"""
WITH iqr_stats AS (
    SELECT
        PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY revenue) AS q1,
        PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY revenue) AS q3,
        PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY revenue) -
        PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY revenue) AS iqr
    FROM transactions
)
SELECT
    CASE
        WHEN revenue < q1 - {cfg['sql_analytics']['iqr_multiplier']} * iqr
          OR revenue > q3 + {cfg['sql_analytics']['iqr_multiplier']} * iqr
        THEN 'outlier' ELSE 'normal'
    END AS classification,
    COUNT(*) AS n,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM transactions), 2) AS pct
FROM transactions, iqr_stats
GROUP BY 1
"""
print('=== SQL IQR Outlier Detection ===')
display(con.execute(outlier_sql).df())

# SQL: Category grouped stats
cat_sql = """
SELECT
    category,
    COUNT(*) AS n,
    ROUND(AVG(revenue), 2) AS mean,
    ROUND(STDDEV_SAMP(revenue), 2) AS std,
    ROUND(PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY revenue), 2) AS median,
    ROUND(SUM(revenue), 2) AS total_revenue
FROM transactions
GROUP BY category
ORDER BY mean DESC
"""
print('\n=== SQL Grouped Statistics by Category ===')
display(con.execute(cat_sql).df())

In [ ]:
# SQL: Linear regression REGR_ functions
spend = rng2.uniform(100, 2000, n_txn)
rev_reg = 500 + 3.5 * spend + rng2.normal(0, 200, n_txn)
reg_df = pd.DataFrame({'ad_spend': spend, 'revenue': rev_reg})
con.register('ad_data', reg_df)

reg_sql = """
SELECT
    ROUND(REGR_SLOPE(revenue, ad_spend), 4)      AS slope,
    ROUND(REGR_INTERCEPT(revenue, ad_spend), 2)  AS intercept,
    ROUND(REGR_R2(revenue, ad_spend), 4)         AS r_squared,
    ROUND(CORR(revenue, ad_spend), 4)            AS pearson_r,
    REGR_COUNT(revenue, ad_spend)                AS n
FROM ad_data
"""
reg_result = con.execute(reg_sql).df()
print('=== SQL Linear Regression: revenue ~ ad_spend ===')
print(f'  True model: revenue = 500 + 3.5 * ad_spend + noise')
display(reg_result)

# Visualise regression
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(spend[:200], rev_reg[:200], alpha=0.4, s=15, color='steelblue', label='Data')
x_line = np.array([spend.min(), spend.max()])
slope = reg_result['slope'].iloc[0]
intercept = reg_result['intercept'].iloc[0]
ax.plot(x_line, intercept + slope * x_line, 'firebrick', lw=2,
        label=f'y = {intercept:.0f} + {slope:.2f}x (R²={reg_result["r_squared"].iloc[0]:.4f})')
ax.set_xlabel('Ad Spend'); ax.set_ylabel('Revenue')
ax.set_title('SQL Linear Regression via DuckDB REGR_ Functions')
ax.legend()
plt.tight_layout()
plt.savefig(cfg['data']['output_dir'] + 'notebook_sql_regression.png', dpi=150)
plt.show()

con.close()
print('\nAll analyses complete!')

## Summary

| Concept | Key Takeaway |
|---|---|
| CLT | Sample means are normal for large n, regardless of the population distribution |
| p-value | P(data this extreme \| H₀ true) — not P(H₀ is true) |
| Cohen's d | Effect size in standard deviation units — use alongside p-value |
| Power | Probability of detecting a real effect; plan sample size before running experiments |
| Bayesian CI | P(θ ∈ CI \| data) = 95% — a direct probability statement about the parameter |
| Frequentist CI | 95% of such intervals would contain the true parameter in repeated sampling |
| Beta-Binomial | Conjugate model for binary outcomes — update belief with each observation |
| P(B > A) | Bayesian probability B is better — incorporates uncertainty in both estimates |
| SQL PERCENTILE_CONT | Exact quantile computation at database scale |
| SQL REGR_* | OLS regression entirely in SQL — no Python/R required |
